# Determine season lengths for the suite of GWLs

Here I will use the GWLs as determined for the parent CMIP6 models that I am using from NEX-GDDP-CMIP6 under the assumption that downscaling keeps the same trends and timing. I will use the central year in the calculated 20-yr periods to get the season length from each model, then use the 10th, 50th, and 90th percentile values to get a central estimate of season length and uncertainty across the full ensemble of models.

In [1]:
import xarray as xr
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.fft import fft

import matplotlib.ticker as mticker
from matplotlib.ticker import AutoMinorLocator, MultipleLocator
from cartopy import crs as ccrs, feature as cfeature
import netCDF4
from netCDF4 import Dataset
from datetime import datetime as dt

import scipy.stats as stats
from sklearn import linear_model
import statsmodels.api as sm
import seaborn as sns

import pymannkendall as mk

# to get rid of runtime warnings when OLS fits hit a divide by zero - handled by setting R2 and p-vals appropriately
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

# latitude & longitude ranges for NH and SH midlatitudes so they will be easy to adjust
nh_min = 23.5
nh_max = 70
sh_min = -60
sh_max = -23.5

## GWLs from Matthias Hauser

In [44]:
# have to clean some extra spaces and ignore some comments in the file
gwls = pd.read_csv("../../../../Data/Other/cmip6_warming_levels_all_ens_1850_1900_grid.csv", comment="#", skipinitialspace=True)
#gwls.columns

# save it out cleaned for reading in the future
gwls.to_csv("../../../../Data/Other/cmip6_warming_levels_all_ens_1850_1900_cleaned.csv", index=False)

#### read in cleaned version

In [45]:
# read in cleaned version
gwls = pd.read_csv("../../../../Data/Other/cmip6_warming_levels_all_ens_1850_1900_cleaned.csv")
gwls

,model,ensemble,exp,grid,warming_level,start_year,end_year
0,CAMS-CSM1-0,r1i1p1f1,ssp119,gn,1.00,2017,2036
1,CAMS-CSM1-0,r2i1p1f1,ssp119,gn,1.00,2032,2051
2,CNRM-ESM2-1,r1i1p1f2,ssp119,gr,1.00,2007,2026
3,CNRM-ESM2-1,r2i1p1f2,ssp119,gr,1.00,2011,2030
4,CNRM-ESM2-1,r3i1p1f2,ssp119,gr,1.00,2009,2028
...,...,...,...,...,...,...,...
10507,UKESM1-0-LL,r1i1p1f2,ssp585,gn,0.61,1994,2013
10508,UKESM1-0-LL,r2i1p1f2,ssp585,gn,0.61,1998,2017
10509,UKESM1-0-LL,r3i1p1f2,ssp585,gn,0.61,1995,2014
10510,UKESM1-0-LL,r4i1p1f2,ssp585,gn,0.61,1996,2015


In [46]:
gwls.describe()

,warming_level,start_year,end_year
count,10512.000000,10512.000000,10512.000000
mean,1.660599,2018.400114,2037.400114
std,0.894835,26.066824,26.066824
min,0.610000,1906.000000,1925.000000
25%,1.000000,1998.000000,2017.000000
50%,1.500000,2014.000000,2033.000000
75%,2.000000,2037.000000,2056.000000
max,4.000000,2081.000000,2100.000000


## Current list of included models

In [58]:
models = ['ACCESS-ESM1-5',
    'BCC-CSM2-MR',
    'CMCC-ESM2',
    'FGOALS-g3',
    'GFDL-ESM4',
    'GISS-E2-1-G',
    'MIROC-ES2L',
    'MIROC6',
    'MPI-ESM1-2-HR',
    'MRI-ESM2-0',
    'NorESM2-LM']

ensembles = ['r1i1p1f1',
    'r1i1p1f1',
    'r1i1p1f1',
    'r3i1p1f1',
    'r1i1p1f1',
    'r1i1p1f2',
    'r1i1p1f2',
    'r1i1p1f1',
    'r1i1p1f1',
    'r1i1p1f1',
    'r1i1p1f1'
    ]

grids = ['gn',
         'gn',
         'gn',
         'gn',
         'gr1',
         'gn',
         'gn',
         'gn',
         'gn',
         'gn',
         'gn'
]

expts = ['historical', 
         'ssp245', 
         'ssp370']

wls = [1.5,2.0,2.5,3.0,4.0]

In [59]:
# reduce list
gwls_red = gwls.query('(model in @models) and (ensemble in @ensembles) and (exp in @expts) and (warming_level in @wls)')
gwls_red = gwls_red.reset_index(drop=True)
gwls_red

,model,ensemble,exp,grid,warming_level,start_year,end_year
0,ACCESS-ESM1-5,r1i1p1f1,ssp245,gn,1.5,2020,2039
1,ACCESS-ESM1-5,r3i1p1f1,ssp245,gn,1.5,2023,2042
2,BCC-CSM2-MR,r1i1p1f1,ssp245,gn,1.5,2026,2045
3,CMCC-ESM2,r1i1p1f1,ssp245,gn,1.5,2021,2040
4,FGOALS-g3,r1i1p1f1,ssp245,gn,1.5,2022,2041
...,...,...,...,...,...,...,...
96,MPI-ESM1-2-HR,r1i1p1f1,ssp370,gn,3.0,2072,2091
97,MPI-ESM1-2-HR,r3i1p1f1,ssp370,gn,3.0,2069,2088
98,MRI-ESM2-0,r1i1p1f1,ssp370,gn,3.0,2064,2083
99,MRI-ESM2-0,r3i1p1f1,ssp370,gn,3.0,2069,2088


In [85]:
gwls_fin = gwls_red.query("index == 0")
gwls_fin

,model,ensemble,exp,grid,warming_level,start_year,end_year
0,ACCESS-ESM1-5,r1i1p1f1,ssp245,gn,1.5,2020,2039


In [82]:
# i=0
# gwls_red.query("(model == @models[@i]) and (ensemble == @ensembles[@i])")

In [86]:
for i in np.arange(0,len(models)):
    gwls_fin = pd.concat([gwls_fin, gwls_red.query("(model == @models[@i]) and (ensemble == @ensembles[@i])")])
    
gwls_fin.drop(index=gwls_fin.index[0], axis=0, inplace=True)
gwls_fin = gwls_fin.reset_index(drop=True)
gwls_fin



,model,ensemble,exp,grid,warming_level,start_year,end_year
0,ACCESS-ESM1-5,r1i1p1f1,ssp370,gn,1.5,2024,2043
1,ACCESS-ESM1-5,r1i1p1f1,ssp245,gn,2.0,2036,2055
2,ACCESS-ESM1-5,r1i1p1f1,ssp370,gn,2.0,2039,2058
3,ACCESS-ESM1-5,r1i1p1f1,ssp245,gn,2.5,2057,2076
4,ACCESS-ESM1-5,r1i1p1f1,ssp370,gn,2.5,2049,2068
...,...,...,...,...,...,...,...
65,NorESM2-LM,r1i1p1f1,ssp245,gn,1.5,2046,2065
66,NorESM2-LM,r1i1p1f1,ssp370,gn,1.5,2042,2061
67,NorESM2-LM,r1i1p1f1,ssp245,gn,2.0,2076,2095
68,NorESM2-LM,r1i1p1f1,ssp370,gn,2.0,2060,2079


### Reduced to only those members I have, save it out

In [87]:
gwls_fin.to_csv("../../../../Data/Other/cmip6_warming_levels_all_ens_1850_1900_cleaned_reduced.csv", index=False)


### Read back in the reduced list for analysis

In [88]:
gwls = pd.read_csv("../../../../Data/Other/cmip6_warming_levels_all_ens_1850_1900_cleaned_reduced.csv")
gwls

,model,ensemble,exp,grid,warming_level,start_year,end_year
0,ACCESS-ESM1-5,r1i1p1f1,ssp370,gn,1.5,2024,2043
1,ACCESS-ESM1-5,r1i1p1f1,ssp245,gn,2.0,2036,2055
2,ACCESS-ESM1-5,r1i1p1f1,ssp370,gn,2.0,2039,2058
3,ACCESS-ESM1-5,r1i1p1f1,ssp245,gn,2.5,2057,2076
4,ACCESS-ESM1-5,r1i1p1f1,ssp370,gn,2.5,2049,2068
...,...,...,...,...,...,...,...
65,NorESM2-LM,r1i1p1f1,ssp245,gn,1.5,2046,2065
66,NorESM2-LM,r1i1p1f1,ssp370,gn,1.5,2042,2061
67,NorESM2-LM,r1i1p1f1,ssp245,gn,2.0,2076,2095
68,NorESM2-LM,r1i1p1f1,ssp370,gn,2.0,2060,2079


## Determine central year and then read in data to get season lengths from each